In [0]:
%pip install sentence-transformers faiss-cpu PyMuPDF requests gradio pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.8/36.8 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 72.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.7/419.7 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.8/433.8 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.8/220.8 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.6/196.6 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 103.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.1/176.1 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.9/542.9 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
dbutils.library.restartPython()

In [0]:
import fitz  # PyMuPDF
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

In [0]:
def extract_pdf_text(binary_content):
    try:
        doc = fitz.open(stream=binary_content, filetype="pdf")
        text = ""
        for page in doc:
            text += page.get_text()
        return text[:50000]  # cap size (IMPORTANT)
    except:
        return ""

In [0]:
extract_udf = udf(extract_pdf_text, StringType())

In [0]:
df_bronze = spark.table("krishi_dhwani.bronze.icar_raw")

In [0]:
df_silver = (
    df_bronze
    .withColumn("text", extract_udf("content"))
    .filter("length(text) > 100")  # remove bad PDFs
    .select("path", "text", "modificationTime")
)

In [0]:
df_silver.write.format("delta").mode("overwrite") \
    .saveAsTable("krishi_dhwani.silver.icar_text")

In [0]:
spark.sql("SELECT COUNT(*) FROM krishi_dhwani.silver.icar_text").show()

+--------+
|COUNT(*)|
+--------+
|       1|
+--------+



In [0]:
spark.sql("""
SELECT path, substring(text, 1, 300) as preview
FROM krishi_dhwani.silver.icar_text
""").show(truncate=False)

+-------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                         |preview                                                                                                                                                                                                                                                                                                               |
+-------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [0]:
spark.sql("""
SELECT COUNT(*) 
FROM krishi_dhwani.silver.icar_text
WHERE length(text) < 50
""").show()

+--------+
|COUNT(*)|
+--------+
|       0|
+--------+



In [0]:
%sql
CREATE OR REPLACE TABLE krishi_dhwani.silver.market_prices AS
SELECT
    date,
    crop,
    mandi,
    ROUND(price_per_quintal / 100.0, 2) AS price_per_kg_inr,
    price_per_quintal,
    unit
FROM krishi_dhwani.bronze.market_prices_raw

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE krishi_dhwani.silver.weather AS
SELECT *
FROM krishi_dhwani.bronze.weather_raw
WHERE forecast_date >= current_date()

num_affected_rows,num_inserted_rows


In [0]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle
import os

/local_disk0/.ephemeral_nfs/envs/pythonEnv-890fc5a1-7b76-4ea2-b975-5275293e061e/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


In [0]:
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [0]:
rows = spark.table("krishi_dhwani.silver.icar_text").collect()

In [0]:
def chunk_text(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

In [0]:
all_chunks = []
all_metadata = []

for row in rows:
    chunks = chunk_text(row["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadata.append({
            "source": row["path"],
            "chunk_id": i
        })

print("Total chunks:", len(all_chunks))

Total chunks: 112


In [0]:
batch_size = 32
embeddings = []

for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings.append(emb)

embeddings = np.vstack(embeddings).astype("float32")

In [0]:
print(embeddings.shape)

(112, 768)


In [0]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)  # cosine similarity
faiss.normalize_L2(embeddings)

index.add(embeddings)

In [0]:
print("Total vectors:", index.ntotal)

Total vectors: 112


In [0]:
faiss.write_index(index, "/tmp/icar.index")

with open("/tmp/metadata.pkl", "wb") as f:
    pickle.dump({
        "chunks": all_chunks,
        "metadata": all_metadata
    }, f)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS krishi_dhwani.silver.faiss_index;

In [0]:
dbutils.fs.mkdirs("/Volumes/krishi_dhwani/silver/faiss_index/")

True

In [0]:
faiss.write_index(index, "/Volumes/krishi_dhwani/silver/faiss_index/icar.index")

In [0]:
import pickle

with open("/Volumes/krishi_dhwani/silver/faiss_index/metadata.pkl", "wb") as f:
    pickle.dump({
        "chunks": all_chunks,
        "metadata": all_metadata
    }, f)

In [0]:
dbutils.fs.ls("/Volumes/krishi_dhwani/silver/faiss_index/")

[FileInfo(path='dbfs:/Volumes/krishi_dhwani/silver/faiss_index/icar.index', name='icar.index', size=344109, modificationTime=1775585456000),
 FileInfo(path='dbfs:/Volumes/krishi_dhwani/silver/faiss_index/metadata.pkl', name='metadata.pkl', size=58531, modificationTime=1775585467000)]